# Auditoría point-in-time — ¿qué se sabía a las 12h de D-1?

**Regla de todo el proyecto:** cada feature solo puede contener lo que se conocía al **cierre de la subasta** del mercado diario (12h de D-1). Este notebook comprueba esa regla **con datos**, fuente a fuente, en vez de suponerla.

**Método (ESIOS):** hacer una "foto" de los valores de las previsiones de mañana **antes** de la subasta (script `scripts/check_publicacion_esios.py`, que guarda CSVs en `data/monitoring/check_esios/`) y compararla con los valores del mismo día descargados **después**. Si una previsión cambia tras la subasta, el histórico de `tabla_ESIOS` (que deduplica con `keep="last"`) guarda la versión posterior → **leakage**.

In [ ]:
from pathlib import Path
import pandas as pd
import sys
sys.path.append("..")
from src.ingesta.esios import descargar_indicador, parsear_indicador, INDICADORES_PREDICTIVO
%load_ext autoreload
%autoreload 2



CARPETA = Path("../data/monitoring/check_esios")
OBJ = "2026-09-29"
USADAS = ["demanda_prevista_diaria", "eolica_prevista_d1", "solar_fv_prevista", "solar_termica_prevista"]

for nombre in USADAS:
    fotos = sorted(CARPETA.glob(f"{nombre}__obj{OBJ}__*.csv"))
    antes = pd.read_csv(fotos[0])                                   # foto de ayer, antes de la subasta
    data = descargar_indicador.fn(nombre, INDICADORES_PREDICTIVO[nombre], f"{OBJ}T00:00:00", f"{OBJ}T23:59:59")
    ahora = parsear_indicador.fn(data, nombre)
    ahora["datetime_utc"] = ahora["datetime_utc"].astype(str)
    cmp = antes.merge(ahora, on="datetime_utc", suffixes=("_antes", "_ahora"))
    diff = (cmp[f"{nombre}_ahora"] - cmp[f"{nombre}_antes"]).abs()
    print(f"{nombre:<25} foto {fotos[0].stem[-13:]} | horas que cambian: {(diff > 0).sum():>2}/{len(cmp)} "
          f"| cambio máx: {diff.max():.1f} | medio: {diff.mean():.1f} | nombre: {data['indicator']['name']}")

## Resultado (foto 28-sep 11:23 vs mañana del 29-sep, día objetivo 29-sep)

| Indicador | Nombre oficial | Horas que cambian tras la subasta | Veredicto |
|---|---|---|---|
| `eolica_prevista_d1` (1777) | Previsión diaria D+1 eólica | **0/24** | ✅ estable: se publica ~10:50 y no se toca |
| `demanda_prevista_diaria` (460) | Previsión diaria de la demanda eléctrica peninsular | 24/24 (±1.527 de media) | ⚠️ se revisa → leakage |
| `solar_fv_prevista` (542) | Generación prevista Solar fotovoltaica | 14/24, las horas de sol (máx 14.812) | ⚠️ se revisa → leakage |
| `solar_termica_prevista` (543) | Generación prevista Solar térmica | 24/24 | ⚠️ se revisa → leakage |

**Asimetría de la evidencia:** para descartar un indicador basta con que cambie **un** día; para dar uno por estable, un día no basta → se siguen tomando fotos (mañana y noche) durante una semana para confirmar la eólica D+1.

**Decisión:** el set predictivo usa **solo** `eolica_prevista_d1`, mediante una **lista blanca** (`_PREVISTAS_D1` en `src/features/feature_sets.py`): ninguna previsión nueva entra al modelo sin verificarse antes. El coste de quitar las otras tres se mide en el notebook 06 (ablación walk-forward). Ver diario 2026-09-29.

**Otras fuentes auditadas:**
- **AEMET:** valores climatológicos **observados**, publicados con ~2 días de retraso → a las 12h de D-1 lo último disponible es D-3 → el predictivo usa la meteo retrasada 3 días (`_lag3d`).
- **ESIOS genéricas** (`demanda_prevista` 544, `eolica_prevista` 541): se refrescan intradía → fuera del modelo.
- **Precio (lags):** `precio_lag_24` / `precio_lag_168` son precios ya casados en subastas anteriores → disponibles.